# Advanced RAG (Re-ranking)

## Task Description
Improve RAG accuracy using a Cross-Encoder Re-ranker.

In this notebook, we will:
1. Install libraries.
2. Import modules.
3. Load Knowledge Base.
4. Create Embeddings & Index.
5. Retrieve Documents.
6. Re-rank Documents.
7. Generate Answer.
8. Save Results.


## License

This notebook is licensed under the Apache License 2.0. See the [LICENSE](https://github.com/huggingface/transformers/blob/main/LICENSE) file for more details.

This notebook focuses on open-source models and is optimized to run on the Google Colab Free Tier (T4 GPU).

## 1. Installation
Install `langchain`, `sentence-transformers`, `faiss-cpu`, `rank_bm25` (optional but good), `torch`.

In [ ]:
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers torch accelerate bitsandbytes

## 2. Imports
Import RAG components.

In [ ]:
import torch
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from sentence_transformers import CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"

## 3. Load Data
Define documents.

In [ ]:
text_data = """
Python is a high-level, general-purpose programming language. Its design philosophy emphasizes code readability.
Java is a high-level, class-based, object-oriented programming language that is designed to have as few implementation dependencies as possible.
C++ is a high-level general-purpose programming language created by Bjarne Stroustrup as an extension of the C programming language.
Rust is a multi-paradigm, general-purpose programming language. Rust emphasizes performance, type safety, and concurrency.
"""
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
docs = text_splitter.create_documents([text_data])

## 4. Data Cleaning
Create Vector Store.

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2", model_kwargs={'device': device})
vector_store = FAISS.from_documents(docs, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 5}) # Retrieve more for re-ranking

## 5. Load Model
Load Re-ranker and Generator.

In [ ]:
# Re-ranker
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device)

# Generator (Small LLM for answer)
model_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16).to(device)

## 6. Inference
Retrieve -> Re-rank -> Generate.

In [ ]:
query = "What language emphasizes performance?"

# 1. Retrieve
retrieved_docs = retriever.invoke(query)

# 2. Re-rank
pairs = [[query, doc.page_content] for doc in retrieved_docs]
scores = cross_encoder.predict(pairs)
top_k_indices = scores.argsort()[::-1][:2] # Keep top 2
top_docs = [retrieved_docs[i] for i in top_k_indices]

print("Top Re-ranked Docs:")
for d in top_docs: print(f"- {d.page_content}")

# 3. Generate
context = "\n".join([d.page_content for d in top_docs])
prompt_text = f"Answer based on context:\n{context}\n\nQuestion: {query}\nAnswer:"
inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=50)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\nFinal Answer: {answer}")

## 7. Save Results
Save answer.

In [ ]:
with open('results.txt', 'w') as f:
    f.write(answer)
print('Results saved.')

## 8. Advanced Usage
Explore Hybrid Search (BM25 + Dense) using `EnsembleRetriever`.

In [ ]:
print("Try Hybrid Search next!")